# Case 1: Diffusion into a Semi-Infinite Medium

For a semi-infinite medium occupying $x>0$, assume:
- The medium is homogeneous and extends indefinitely in the positive $x$ direction.
- The initial potential is zero: $c(x,0)=0$ for $x>0$.
- The potential remains zero far from the surface: $c(x,t)\to 0$ as $x\to\infty$.
- The potential at the surface is suddenly raised to and maintained at $c_0$ for $t>0$.

The analytical solution is

$$
c(x,t) = c_0\,\operatorname{erfc}\left(\frac{x}{2\sqrt{Dt}}\right),
$$

where $\operatorname{erfc}$ is the complementary error function. It satisfies the maintained surface value $c(0,t)=c_0$, while the potential approaches zero with distance into the medium. The characteristic penetration distance grows in proportion to $\sqrt{Dt}$.

The plot uses $D=1$ and $c_0=1$ to show how the potential profile changes with time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from math import erfc


def semi_infinite_constant_surface(x, t, D, c0):
    """Diffusion into x >= 0 from a surface held at potential c0."""
    scaled_distance = np.asarray(x) / (2 * np.sqrt(D * t))
    erfc_values = np.vectorize(erfc)(scaled_distance)
    return c0 * erfc_values

In [ ]:
D = 1.0
c0 = 1.0
times = [0.1, 0.5, 1.0, 2.0]

x = np.linspace(0, 10, 500)

fig, ax = plt.subplots()
for t in times:
    ax.plot(
        x,
        semi_infinite_constant_surface(x, t, D, c0),
        label=f"Dt = {D * t:g}",
    )

ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.axhline(c0, color="black", linewidth=0.8, linestyle=":")
ax.set(
    xlabel="Distance from the surface, x",
    ylabel="Potential c(x, t)",
    title="Diffusion from a constant-potential surface",
    ylim=(-0.05, 1.05),
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Finite interval with a zero-potential boundary

For a finite interval $0<x<L$, keep the surface at $c_0$ and set the potential to zero at $x=L$. Assume the initial potential is zero inside the interval:

$$
c(x,0)=0, \qquad c(0,t)=c_0, \qquad c(L,t)=0.
$$

The steady-state profile is linear, $c_0(1-x/L)$. Adding the decaying transient modes gives

$$
c(x,t)=c_0\left(1-\frac{x}{L}\right)-\frac{2c_0}{\pi}\sum_{n=1}^{\infty}\frac{1}{n}\sin\left(\frac{n\pi x}{L}\right)\exp\left(-\frac{n^2\pi^2Dt}{L^2}\right).
$$

This solution maintains $c_0$ at $x=0$ and zero potential at $x=L$; as time increases, the profile approaches its linear steady state.

In [ ]:
def finite_interval_constant_surface(x, t, D, L, c0, n_terms=200):
    """Diffusion on 0 <= x <= L with fixed potentials at both boundaries."""
    x = np.atleast_1d(np.asarray(x, dtype=float))
    modes = np.arange(1, n_terms + 1)
    steady_state = c0 * (1 - x / L)
    spatial_modes = np.sin(np.outer(modes * np.pi / L, x))
    decay = np.exp(-(modes**2) * np.pi**2 * D * t / L**2)
    amplitudes = -2 * c0 / (np.pi * modes)
    transient = np.sum(amplitudes[:, None] * spatial_modes * decay[:, None], axis=0)
    return steady_state + transient

In [ ]:
D = 1.0
L = 4.0
c0 = 1.0
times_finite = [0.1, 0.5, 1.0, 2.0]

x_finite = np.linspace(0, L, 500)

fig, ax = plt.subplots()
for t in times_finite:
    ax.plot(
        x_finite,
        finite_interval_constant_surface(x_finite, t, D, L, c0),
        label=f"Dt = {D * t:g}",
    )

ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.axhline(c0, color="black", linewidth=0.8, linestyle=":")
ax.set(
    xlabel="Distance from the surface, x",
    ylabel="Potential c(x, t)",
    title="Diffusion with a zero-potential boundary at x = L",
    xlim=(0, L),
    ylim=(-0.05, 1.05),
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()